<a href="https://colab.research.google.com/github/THEJoshinator20/ST554-HW/blob/main/554_HW7_McClure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Homework 7 ST-554

By Joshua McClure

Library for Entire Homework

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV, ElasticNetCV
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import log_loss, accuracy_score
from sklearn.pipeline import Pipeline

##Goal

The purpose of this homework is to practice fitting MLR and logistic regression models (including penalized or regularized models).

##Data

We will use a dataset from the UCI Machine Learning Repository. Which covers red and white variants of the Portuguese "Vinho Verde" wine. We will use the datasets winequality-red.csv winequality-white.csv and ignore winequality.names.

##To Do:
Create a document that goes through your process of reading the data, combining it, manipulating/creating
any variables, and fitting and choosing a final model for both the multiple linear regression modeling and
the logistic regression modeling described below.

##Step 1: Read in and Combine Data

* Read in the winequality-red.csv and winequality-white.csv files available on the UCI machine learning repository site.
* Combine these two datasets and create a new variable that represents the type of wine (red or white)

In [4]:
# Read in the red wine dataset
red_wine = pd.read_csv('winequality-red.csv')
red_wine['wine_type'] = 'red'

# Read in the white wine dataset
white_wine = pd.read_csv('winequality-white.csv')
white_wine['wine_type'] = 'white'

# Combine the two datasets
df = pd.concat([red_wine, white_wine], ignore_index=True)

# Display the first few rows of the combined DataFrame
print("Combined DataFrame head:")
print(df.head())

# Display variable names and dtypes
print("\nCombined DataFrame info:")
df.info()

# Displays value counts for each wine type in combined datset
print("\nValue counts for wine_type:")
print(df['wine_type'].value_counts())

Combined DataFrame head:
  fixed acidity;"volatile acidity";"citric acid";"residual sugar";"chlorides";"free sulfur dioxide";"total sulfur dioxide";"density";"pH";"sulphates";"alcohol";"quality"  \
0   7.4;0.7;0;1.9;0.076;11;34;0.9978;3.51;0.56;9.4;5                                                                                                                        
1   7.8;0.88;0;2.6;0.098;25;67;0.9968;3.2;0.68;9.8;5                                                                                                                        
2  7.8;0.76;0.04;2.3;0.092;15;54;0.997;3.26;0.65;...                                                                                                                        
3  11.2;0.28;0.56;1.9;0.075;17;60;0.998;3.16;0.58...                                                                                                                        
4   7.4;0.7;0;1.9;0.076;11;34;0.9978;3.51;0.56;9.4;5                                                          

The red and white wine datsets are read in seperately and combined together. Then to double check for any potential errors the head, info, and value counts for red and white wine were displayed. No errors or missing variables were shown which means that the combination was successful.


---

##Step 2: Split the Data
* Split up the data set into a training and test set. For this, I want you to use stratified sampling to
make sure that you have a similar proportion of white and red wines in the training and test sets. This
can be done with the train_test_split() function.

In [5]:
# --- Step 1: Fix Column Names ---

# Some slight errors were made in the column names which is fixed with semicolon and quote removal
original_columns_nme = df.columns[0].replace('"', '')
column_names = original_columns_nme.split(';')

# Apply a string split to the first column and expand into new columns
analyzed_data = df[df.columns[0]].str.split(';', expand=True)
analyzed_data.columns = column_names

# Convert analyzed columns to numeric, handling potential errors
for col in analyzed_data.columns:
    analyzed_data[col] = pd.to_numeric(analyzed_data[col], errors='coerce')

# Clean up column names for analyzed columns
analyzed_data.columns = [col.strip().replace(' ', '_') for col in analyzed_data.columns]

# Apply corrections to combined dataset and make sure wine type column is in combined dataset
comb_df = pd.concat([analyzed_data, df['wine_type']], axis=1)

# --- Step 2: Define X and Y / Training and Test Data Split ---

# Define features (X) and target (y)
X = comb_df.drop(columns=['quality', 'wine_type'])
y = comb_df['quality']

# Perform the stratified train-test split using the stratify in wine type to ensure
# approximately similar proportions of red and white wine
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, # 20% for testing, 80% for training
    random_state=42, # for reproducibility
    stratify=comb_df['wine_type'])

# --- Step 3: Display Output ---

# Printing of the train-test data shapes for X and Y
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

# Proportion of wine types in the original combined dataframe
print("\nDistribution of wine_type in original comb_df:")
print(comb_df['wine_type'].value_counts(normalize=True))

# Proportion of wine types in the Training dataframe
print("\nDistribution of wine_type in training set:")
print(comb_df.loc[X_train.index, 'wine_type'].value_counts(normalize=True))

# Proportion of wine types in the Test dataframe
print("\nDistribution of wine_type in test set:")
print(comb_df.loc[X_test.index, 'wine_type'].value_counts(normalize=True))

Shape of X_train: (5197, 11)
Shape of X_test: (1300, 11)
Shape of y_train: (5197,)
Shape of y_test: (1300,)

Distribution of wine_type in original comb_df:
wine_type
white    0.753886
red      0.246114
Name: proportion, dtype: float64

Distribution of wine_type in training set:
wine_type
white    0.753896
red      0.246104
Name: proportion, dtype: float64

Distribution of wine_type in test set:
wine_type
white    0.753846
red      0.246154
Name: proportion, dtype: float64


The datset columns had issues regarding the column names that were resolved in step 1.

Then in step 2 the defining of X and Y along with the train-test split code was done while ensuring repeatability and wine type proportion were constant.

Finally, step 3 displays the output for step 2 while also displaying the consistancy of proportions across the different dataframes.

---

##Regression Task (alcohol as Response)

###Train Models
* Fit four different multiple linear regression models.
  * At least one should include interaction terms
  * At least one should include some polynomial terms (you may want to standardize your predictors
but that is up to you)
  * Use CV to select your best MLR model
* Fit a LASSO model with a set of predictors of your choosing
  * Use at least five predictors
  * Use CV to select the tuning parameter
* Fit a Ridge Regression model with a set of predictors of your choosing
  * Use at least five predictors
  * Use CV to select the tuning parameter
* Fit an Elastic Net model with a set of predictors of your choosing
  * Use at least five predictors
  * Use CV to select the tuning parameters

In [6]:
# --- Step 1: Prepare data for Regression Task (alcohol as Response) ---

# Creation of X and Y variables for MLR models
# X variable needs to not include 'alcohol', 'quality', 'wine_type' columns
# Y variable needs to include 'alcohol' column
X_reg = comb_df.drop(columns=['alcohol', 'quality', 'wine_type'])
y_reg = comb_df['alcohol']

# Perform stratified train-test split for regression using previous method
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg,
    test_size = 0.2,
    random_state = 42,
    stratify = comb_df['wine_type'])

# Standardize features of X for the train and test dataframes
scaler = StandardScaler()
X_train_reg_scaled = scaler.fit_transform(X_train_reg)
X_test_reg_scaled = scaler.transform(X_test_reg)

# Put X Train and X Test back into Dataframe to ensure consitency of data and of column names
X_train_reg_scaled = pd.DataFrame(X_train_reg_scaled, columns = X_train_reg.columns, index = X_train_reg.index)
X_test_reg_scaled = pd.DataFrame(X_test_reg_scaled, columns = X_test_reg.columns, index = X_test_reg.index)

# Test print of datasets to ensure consistency of shape and the proper retention of columns
print("Shapes of regression datasets after preparation:")
print(f"X_train_reg_scaled: {X_train_reg_scaled.shape}")
print(f"y_train_reg: {y_train_reg.shape}")
print(f"X_test_reg_scaled: {X_test_reg_scaled.shape}")
print(f"y_test_reg: {y_test_reg.shape}\n")

# Set up cross-validation (5-fold) for future usage
cv = KFold(n_splits = 5, shuffle = True, random_state = 42)

# Dictionary to store all fitted MLR models and their Cross-Validation scores for selection
mlr_candidates = {}

# --- Step 2: Train Four Different Multiple Linear Regression Models ---

# Model 1: Base MLR
mlr_base = LinearRegression()
scores_base = cross_val_score(mlr_base, X_train_reg_scaled, y_train_reg, cv = cv,
                              scoring = 'neg_mean_squared_error', n_jobs = -1)
mlr_base_fitted = mlr_base.fit(X_train_reg_scaled, y_train_reg)
mlr_candidates['Base MLR'] = {'model': mlr_base_fitted, 'cv_score': np.mean(scores_base)}
print(f"Base MLR CV Score (neg_mse): {np.mean(scores_base):.4f}")

# Model 2: MLR with Polynomial Terms (degree 2)
pipeline_mlr_poly = Pipeline([
    ('poly', PolynomialFeatures(degree = 2, include_bias = False)),
    ('regressor', LinearRegression())])

scores_poly = cross_val_score(pipeline_mlr_poly, X_train_reg_scaled, y_train_reg,
                              cv = cv, scoring = 'neg_mean_squared_error', n_jobs=-1)
mlr_poly_fitted = pipeline_mlr_poly.fit(X_train_reg_scaled, y_train_reg)
mlr_candidates['MLR with Poly (deg 2)'] = {'model': mlr_poly_fitted, 'cv_score': np.mean(scores_poly)}
print(f"MLR Poly (deg 2) CV Score (neg_mse): {np.mean(scores_poly):.4f}")

# Model 3: MLR with Interaction Terms (degree 2, interaction_only)
pipeline_mlr_inter = Pipeline([
    ('poly', PolynomialFeatures(degree = 2, interaction_only = True, include_bias = False)),
    ('regressor', LinearRegression())])

scores_inter = cross_val_score(pipeline_mlr_inter, X_train_reg_scaled, y_train_reg, cv = cv,
                               scoring = 'neg_mean_squared_error', n_jobs=-1)
mlr_inter_fitted = pipeline_mlr_inter.fit(X_train_reg_scaled, y_train_reg)
mlr_candidates['MLR with Interactions'] = {'model': mlr_inter_fitted, 'cv_score': np.mean(scores_inter)}
print(f"MLR with Interactions CV Score (neg_mse): {np.mean(scores_inter):.4f}")

# Model 4: MLR with Polynomial and Interaction Terms (degree 3)
pipeline_mlr_poly_complex = Pipeline([
    ('poly', PolynomialFeatures(degree=3, include_bias=False)),
    ('regressor', LinearRegression())])

scores_poly_complex = cross_val_score(pipeline_mlr_poly_complex, X_train_reg_scaled, y_train_reg,
                                      cv = cv, scoring = 'neg_mean_squared_error', n_jobs=-1)
mlr_poly_complex_fitted = pipeline_mlr_poly_complex.fit(X_train_reg_scaled, y_train_reg)
mlr_candidates['MLR with Poly (deg 3)'] = {'model': mlr_poly_complex_fitted, 'cv_score': np.mean(scores_poly_complex)}
print(f"MLR Poly (deg 3) CV Score (neg_mse): {np.mean(scores_poly_complex):.4f}\n")

# Select the best MLR model based on the highest (least negative) CV score
best_mlr_name = max(mlr_candidates, key = lambda k: mlr_candidates[k]['cv_score'])
best_mlr_model = mlr_candidates[best_mlr_name]['model']
print(f"Best MLR Model (based on CV neg_mean_squared_error): {best_mlr_name} with score {mlr_candidates[best_mlr_name]['cv_score']:.4f}\n")

# --- Step 3: Train Regularized Regression Models ---

# LASSO Model (at least five predictors - all 10 features are used)
lasso_model = LassoCV(cv = cv, random_state = 42, n_jobs = -1, max_iter=10000).fit(X_train_reg_scaled, y_train_reg)
print(f"Lasso optimal alpha: {lasso_model.alpha_:.4f}")

# Ridge Regression Model (at least five predictors)
ridge_alphas = np.logspace(-3, 3, 100) # Range of alphas to search
ridge_model = RidgeCV(alphas = ridge_alphas, cv = cv, scoring =
                      'neg_mean_squared_error').fit(X_train_reg_scaled, y_train_reg)
print(f"Ridge optimal alpha: {ridge_model.alpha_:.4f}")

# Elastic Net Model (at least five predictors)
elas_alphas = np.logspace(-3, 3, 10)
elas_l1_ratios = np.linspace(0.1, 0.9, 9)
elas_model = ElasticNetCV(alphas = elas_alphas, l1_ratio = elas_l1_ratios, cv = cv, random_state = 42,
                          n_jobs = -1, max_iter=10000).fit(X_train_reg_scaled, y_train_reg)
print(f"Elastic Net optimal alpha: {elas_model.alpha_:.4f}, l1_ratio: {elas_model.l1_ratio_:.4f}\n")

Shapes of regression datasets after preparation:
X_train_reg_scaled: (5197, 10)
y_train_reg: (5197,)
X_test_reg_scaled: (1300, 10)
y_test_reg: (1300,)

Base MLR CV Score (neg_mse): -0.3100
MLR Poly (deg 2) CV Score (neg_mse): -0.2142
MLR with Interactions CV Score (neg_mse): -0.3003
MLR Poly (deg 3) CV Score (neg_mse): -0.4031

Best MLR Model (based on CV neg_mean_squared_error): MLR with Poly (deg 2) with score -0.2142

Lasso optimal alpha: 0.0009
Ridge optimal alpha: 2.1544
Elastic Net optimal alpha: 0.0010, l1_ratio: 0.9000



###Test Models
* Using your four selected models, compare their performance on the test set.
  * Do so using RMSE as your model metric
  * Do so using MAE as your model metric

In [7]:
# Neatly organized values from MLR Model needed for RMSE and MAE comparisons
final_regression_models_for_testing = {'Base MLR': mlr_candidates['Base MLR']['model'],
    'MLR with Poly (deg 2)': mlr_candidates['MLR with Poly (deg 2)']['model'],
    'MLR with Interactions': mlr_candidates['MLR with Interactions']['model'],
    'MLR with Poly (deg 3)': mlr_candidates['MLR with Poly (deg 3)']['model'],
    'Lasso': lasso_model,
    'Ridge': ridge_model,
    'Elastic Net': elas_model}

# Dictionary to store performance metrics
performance_metrics = {}

# Iterate through each trained model
for model_name, model in final_regression_models_for_testing.items():
    # Handle pipelines correctly when predicting
    if isinstance(model, Pipeline):
        # If the model contains the polynomial or linear regression
        y_pred = model.predict(X_test_reg_scaled)
    else:
        # For other models like Lasso, Ridge, ElasticNet, and Base MLR, they were directly fit on scaled data
        y_pred = model.predict(X_test_reg_scaled)

    # Calculate RMSE (Root Mean Squared Estimator)
    rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred))
    # Calculate MAE (Mean Absolute Error)
    mae = mean_absolute_error(y_test_reg, y_pred)

    # Fill in performance metrics dictionary
    performance_metrics[model_name] = {'RMSE': rmse, 'MAE': mae}

# Convert dictionary to DataFrame for better readability
performance_df = pd.DataFrame(performance_metrics).T # Transpose to have models as rows
performance_df = performance_df.sort_values(by='RMSE')

print("Regression Model Performance on Test Set:")
display(performance_df)

Regression Model Performance on Test Set:


,RMSE,MAE
MLR with Poly (deg 3),0.430215,0.313567
MLR with Poly (deg 2),0.482199,0.347175
MLR with Interactions,0.507985,0.367757
Base MLR,0.527935,0.402527
Ridge,0.528155,0.402743
Lasso,0.528265,0.402706
Elastic Net,0.528304,0.402751


##Classification Task (Wine Type as Response)
* Repeat the training and testing done previously but use logistic regression models.
* Use log-loss or negative log-loss as your metric for choosing models during the training process
* During the testing portion, compare your models on both log-loss and accuracy

##Classification Task Train

In [8]:
# --- Step 1: Prepare data for Classification Task (Wine Type as Response) ---

# Define features (X_clf) and target (y_clf)
# X_clf should exclude 'wine_type' (the new target), 'quality' (old target), and 'alcohol' (regression target)
X_clf = comb_df.drop(columns=['wine_type', 'quality', 'alcohol'])
y_clf = comb_df['wine_type']

# Encode 'wine_type' to numerical (e.g., 'red' -> 0, 'white' -> 1)
y_clf_encoded = y_clf.map({'red': 0, 'white': 1})

# Perform stratified train-test split for classification
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_clf_encoded)

# Standardize features
scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)

# Convert scaled arrays back to DataFrames for consistency
X_train_clf_scaled = pd.DataFrame(X_train_clf_scaled, columns=X_train_clf.columns, index=X_train_clf.index)
X_test_clf_scaled = pd.DataFrame(X_test_clf_scaled, columns=X_test_clf.columns, index=X_test_clf.index)

print("Shapes of classification datasets after preparation:")
print(f"X_train_clf_scaled: {X_train_clf_scaled.shape}")
print(f"y_train_clf: {y_train_clf.shape}")
print(f"X_test_clf_scaled: {X_test_clf_scaled.shape}")
print(f"y_test_clf: {y_test_clf.shape}\n")

# Set up cross-validation (5-fold)
cv_clf = KFold(n_splits=5, shuffle=True, random_state=42)
scoring_clf = 'neg_log_loss' # Metric for CV selection

# Dictionary to store all fitted Logistic Regression models and their CV scores for selection
clf_candidates = {}

# --- Step 2: Train Four Different Logistic Regression Models ---

# Model 1: Base Logistic Regression
log_reg_base = LogisticRegression(random_state = 42, solver = 'liblinear', max_iter = 2000)
scores_log_reg_base = cross_val_score(log_reg_base, X_train_clf_scaled, y_train_clf, cv = cv_clf,
                                       scoring = scoring_clf, n_jobs = -1)
log_reg_base_fitted = log_reg_base.fit(X_train_clf_scaled, y_train_clf)
clf_candidates['Base Logistic Regression'] = {'model': log_reg_base_fitted, 'cv_score': np.mean(scores_log_reg_base)}
print(f"Base Logistic Regression CV Score (neg_log_loss): {np.mean(scores_log_reg_base):.4f}")

# Model 2: Logistic Regression with Polynomial Terms (degree 2)
pipeline_log_reg_poly = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('log_reg', LogisticRegression(random_state=42, solver='liblinear', max_iter=2000))])

scores_log_reg_poly = cross_val_score(pipeline_log_reg_poly, X_train_clf_scaled, y_train_clf,
                                      cv = cv_clf, scoring = scoring_clf, n_jobs = -1)
pipeline_log_reg_poly_fitted = pipeline_log_reg_poly.fit(X_train_clf_scaled, y_train_clf)
clf_candidates['Logistic Regression with Poly (deg 2)'] = {'model': pipeline_log_reg_poly_fitted, 'cv_score': np.mean(scores_log_reg_poly)}
print(f"Logistic Regression Poly (deg 2) CV Score (neg_log_loss): {np.mean(scores_log_reg_poly):.4f}")

# Model 3: Logistic Regression with Interaction Terms (degree 2)
pipeline_log_reg_inter = Pipeline([
    ('poly', PolynomialFeatures(degree=2, interaction_only = True, include_bias = False)),
    ('log_reg', LogisticRegression(random_state = 42, solver = 'liblinear', max_iter = 2000))])

scores_log_reg_inter = cross_val_score(pipeline_log_reg_inter, X_train_clf_scaled, y_train_clf,
                                       cv = cv_clf, scoring = scoring_clf, n_jobs = -1)
pipeline_log_reg_inter_fitted = pipeline_log_reg_inter.fit(X_train_clf_scaled, y_train_clf)
clf_candidates['Logistic Regression with Interactions'] = {'model': pipeline_log_reg_inter_fitted, 'cv_score': np.mean(scores_log_reg_inter)}
print(f"Logistic Regression with Interactions CV Score (neg_log_loss): {np.mean(scores_log_reg_inter):.4f}")

# Model 4: Logistic Regression with Polynomial and Interaction Terms (degree 3)
pipeline_log_reg_poly_complex = Pipeline([
    ('poly', PolynomialFeatures(degree = 3, include_bias = False)),
    ('log_reg', LogisticRegression(random_state = 42, solver='liblinear', max_iter = 5000))])

scores_log_reg_poly_complex = cross_val_score(pipeline_log_reg_poly_complex, X_train_clf_scaled,
                                              y_train_clf, cv=cv_clf, scoring = scoring_clf, n_jobs = -1)
pipeline_log_reg_poly_complex_fitted = pipeline_log_reg_poly_complex.fit(X_train_clf_scaled, y_train_clf)
clf_candidates['Logistic Regression with Poly (deg 3)'] = {'model': pipeline_log_reg_poly_complex_fitted, 'cv_score': np.mean(scores_log_reg_poly_complex)}
print(f"Logistic Regression Poly (deg 3) CV Score (neg_log_loss): {np.mean(scores_log_reg_poly_complex):.4f}\n")

# Select the best of the four initial Logistic Regression models based on the highest (least negative) CV score
best_log_reg_name = max(clf_candidates, key = lambda k: clf_candidates[k]['cv_score'])
best_log_reg_model = clf_candidates[best_log_reg_name]['model']
print(f"Best Initial Logistic Regression Model (based on CV neg_log_loss): {best_log_reg_name} with score {clf_candidates[best_log_reg_name]['cv_score']:.4f}\n")

# --- Step 3: Train Regularized Logistic Regression Models ---

# LASSO Logistic Regression Model (L1 penalty)
# Cs = inverse of regularization strength; smaller C = stronger regularization
lasso_log_reg = LogisticRegressionCV(Cs = 10, penalty = 'l1', solver = 'liblinear',
                                     cv = cv_clf, scoring = scoring_clf, random_state = 42,
                                     n_jobs = -1, max_iter = 2000).fit(X_train_clf_scaled, y_train_clf)
print(f"Lasso Logistic Regression optimal C: {lasso_log_reg.C_[0]:.4f}")

# Ridge Logistic Regression Model (L2 penalty)
ridge_log_reg = LogisticRegressionCV(Cs = 10, penalty = 'l2', solver = 'liblinear',
                                     cv = cv_clf, scoring = scoring_clf, random_state = 42,
                                     n_jobs = -1, max_iter = 2000).fit(X_train_clf_scaled, y_train_clf)
print(f"Ridge Logistic Regression optimal C: {ridge_log_reg.C_[0]:.4f}")

# Elastic Net Logistic Regression Model
elas_log_reg_l1_ratios = np.linspace(0.1, 0.9, 9)
elas_log_reg = LogisticRegressionCV(Cs = 10, l1_ratios = elas_log_reg_l1_ratios,
                                    penalty = 'elasticnet', solver = 'saga', cv = cv_clf,
                                    scoring = scoring_clf, random_state = 42, n_jobs = -1,
                                    max_iter = 5000).fit(X_train_clf_scaled, y_train_clf)
print(f"Elastic Net Logistic Regression optimal C: {elas_log_reg.C_[0]:.4f}, l1_ratio: {elas_log_reg.l1_ratio_[0]:.4f}\n")

Shapes of classification datasets after preparation:
X_train_clf_scaled: (5197, 10)
y_train_clf: (5197,)
X_test_clf_scaled: (1300, 10)
y_test_clf: (1300,)

Base Logistic Regression CV Score (neg_log_loss): -0.0447
Logistic Regression Poly (deg 2) CV Score (neg_log_loss): -0.0386
Logistic Regression with Interactions CV Score (neg_log_loss): -0.0356
Logistic Regression Poly (deg 3) CV Score (neg_log_loss): -0.0596

Best Initial Logistic Regression Model (based on CV neg_log_loss): Logistic Regression with Interactions with score -0.0356

Lasso Logistic Regression optimal C: 2.7826
Ridge Logistic Regression optimal C: 2.7826
Elastic Net Logistic Regression optimal C: 2.7826, l1_ratio: 0.1000



##Classification Task Test

In [9]:
final_classification_models_for_testing = {
    'Base Logistic Regression': clf_candidates['Base Logistic Regression']['model'],
    'Logistic Regression with Poly (deg 2)': clf_candidates['Logistic Regression with Poly (deg 2)']['model'],
    'Logistic Regression with Interactions': clf_candidates['Logistic Regression with Interactions']['model'],
    'Logistic Regression with Poly (deg 3)': clf_candidates['Logistic Regression with Poly (deg 3)']['model'],
    'Lasso Logistic Regression': lasso_log_reg,
    'Ridge Logistic Regression': ridge_log_reg,
    'Elastic Net Logistic Regression': elas_log_reg}

# Dictionary to store performance metrics for classification
classification_performance_metrics = {}

# Iterate through each trained classification model
for model_name, model in final_classification_models_for_testing.items():
    # Predict probabilities for log-loss
    if isinstance(model, Pipeline):
        y_pred_proba = model.predict_proba(X_test_clf_scaled)
        y_pred_class = model.predict(X_test_clf_scaled)
    else:
        y_pred_proba = model.predict_proba(X_test_clf_scaled)
        y_pred_class = model.predict(X_test_clf_scaled)

    # Calculate Log-Loss
    current_log_loss = log_loss(y_test_clf, y_pred_proba)
    # Calculate Accuracy
    current_accuracy = accuracy_score(y_test_clf, y_pred_class)

    classification_performance_metrics[model_name] = {'Log-Loss': current_log_loss, 'Accuracy': current_accuracy}

# Convert dictionary to DataFrame for better readability
classification_performance_df = pd.DataFrame(classification_performance_metrics).T
classification_performance_df = classification_performance_df.sort_values(by='Log-Loss')

print("Classification Model Performance on Test Set:")
display(classification_performance_df)

Classification Model Performance on Test Set:


,Log-Loss,Accuracy
Logistic Regression with Poly (deg 2),0.014215,0.996154
Logistic Regression with Interactions,0.020611,0.993846
Lasso Logistic Regression,0.028053,0.993846
Elastic Net Logistic Regression,0.028271,0.993846
Ridge Logistic Regression,0.028447,0.993846
Base Logistic Regression,0.029402,0.993846
Logistic Regression with Poly (deg 3),0.044305,0.994615
